# 5. Association Rule Mining and ANN Synthesis
Rules describe environmental and road conditions that co-occur with severe collisions. Association metrics reflect empirical correlation, not direct causation. Neural network benchmark evaluation and loss curves are integrated in notebook 02 and the model diagnostic dashboard.

In [2]:
from pathlib import Path
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data/processed/cleaned_accidents.csv')
rule_data = pd.DataFrame({
    'weather': df.weather_conditions.astype(str),
    'light': df.light_conditions.astype(str),
    'road_type': df.road_type.astype(str),
    'speed_band': pd.cut(df.speed_limit, bins=[0,30,50,70,200], labels=['low','medium','high','very_high']).astype(str),
    'area': df.urban_or_rural_area.astype(str),
    'severity': df.severity
}).sample(min(30000, len(df)), random_state=42)
basket = pd.get_dummies(rule_data.astype(str), prefix_sep='=')
frequent = apriori(basket, min_support=.001, use_colnames=True, max_len=3)
rules = association_rules(frequent, metric='confidence', min_threshold=.02)
target_classes = {'severity=Serious', 'severity=Fatal'}
rules = rules[
    rules['consequents'].apply(lambda items: len(items) == 1 and next(iter(items)) in target_classes) &
    rules['antecedents'].apply(lambda items: all(not str(item).startswith('severity=') for item in items))
]
rules = rules[rules.lift > 1].sort_values(['lift', 'support'], ascending=False).head(10)
rules.to_csv(ROOT / 'outputs/tables/association_rules.csv', index=False)
display(rules[['antecedents','consequents','support','confidence','lift']])
print('Rules are descriptive associations from the 30,000-record mining sample. Lift above 1 indicates positive co-occurrence, not statistical significance or causation.')


                                    antecedents  ...      lift
555             frozenset({light=6, weather=1})  ...  3.340234
2690      frozenset({speed_band=high, light=6})  ...  3.270489
230                        frozenset({light=6})  ...  2.909913
2666          frozenset({light=6, road_type=6})  ...  2.816103
2743               frozenset({light=6, area=2})  ...  2.793102
3042  frozenset({speed_band=high, road_type=6})  ...  2.727891
800     frozenset({speed_band=high, weather=1})  ...  2.531191
3232       frozenset({speed_band=high, area=2})  ...  2.443972
320                frozenset({speed_band=high})  ...  2.409322
2213      frozenset({light=1, speed_band=high})  ...  2.178186

[10 rows x 5 columns]
Rules are descriptive associations from the 30,000-record mining sample. Lift above 1 indicates positive co-occurrence, not statistical significance or causation.


## ANN note
Notebook 02 trains the MLPClassifier with two hidden layers and a ReLU activation. Its evaluation appears alongside the classical models so the comparison uses the same held-out data. A higher training score alone would not be enough; we compare the held-out recall and macro F1.